In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import boxcox
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.nn.utils.parametrizations import weight_norm
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
class OutputCrop1d(nn.Module):
    def __init__(self, crop_size: int):
        super().__init__()
        self.crop_size = crop_size

    def forward(self, x: torch.Tensor):
        return x[:, :, :-self.crop_size].contiguous()


class TemporalConvUnit(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        padding: int,
        dilation: int,
        stride: int = 1,
        dropout: float = 0.2,
        name: str | None = None
    ):
        super().__init__()
        self.name = name
        
        # Weight normalisation: https://arxiv.org/abs/1602.07868
        self.conv = weight_norm(
            nn.Conv1d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                padding=padding,
                dilation=dilation,
                stride=stride,
            )
        )
        self.conv.weight.data.normal_(0, 0.01)
        self.crop = OutputCrop1d(padding)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv, self.crop, self.relu, self.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)
    

class TemporalConvBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        padding: int,
        dilation: int,
        stride: int = 1,
        dropout: float = 0.2,
    ):
        """
        :param in_channels: Number of input channels.
            Corresponds to the number of features at each timestep in the input series.
        :param out_channels: Number of output channels.
            Corresponds to the number of features at each timestep in the output series.
        :param kernel_size: Number of weights per filter.
        :param padding: Size of padding to apply to both sides of the input
        """
        super().__init__()
        
        self.unit1 = TemporalConvUnit(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding,
            stride=stride,
            dropout=dropout,
        )
        self.unit2 = TemporalConvUnit(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding,
            stride=stride,
            dropout=dropout,
        )
        self.net = nn.Sequential(self.unit1, self.unit2)

        # Residual connection
        if in_channels != out_channels:
            self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=1)
            self.conv.weight.data.normal_(0, 0.01)
        else:
            self.conv = None
        
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.net(x)
        res = x if self.conv is None else self.conv(x)
        return self.relu(out + res)
    

class TemporalConvNetDecoder(nn.Module):
    def __init__(self, in_features: int, horizon: int = 1):
        super().__init__()
        """
        Linear decoder that maps the final hidden representation from the TCN 
        into the target forecasting horizon.

        :param in_features: Number of input features (channels) from the final TCN layer. 
            This corresponds to the number of learned feature maps at the last timestep.

        :param horizon: Number of future timesteps to predict. 
            The decoder outputs one value per step in the forecast horizon.
        """
        self.linear = nn.Linear(in_features=in_features, out_features=horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x)


class TemporalConvNet(nn.Module):
    def __init__(
        self,
        in_features: int,
        out_features: list[int],
        horizon: int, 
        kernel_size: int = 2,
        dropout: float = 0.2
    ):
        """
        :param in_features: Number of input features at each timestep in the time series.
            This corresponds to the number of input channels to the first convolutional layer.
        
        :param out_features: List specifying the number of output feature maps (channels) 
            for each temporal convolutional block in the network.
            For example, [16, 32, 64] creates three stacked convolutional blocks with
            16, 32, and 64 output channels, respectively.
        
        :param horizon: Number of future timesteps to predict i.e. the forecasting horizon
        
        :param kernel_size: Size of the temporal convolution kernel.
            Controls the receptive field of each convolutional layer.
        
        :param dropout: Dropout probability applied after each convolutional layer.
        """
        super().__init__()
        
        layers = []
        n_layers = len(out_features)
        for i in range(n_layers):
            in_channels = in_features if i == 0 else out_features[i - 1]
            out_channels = out_features[i]
            dilation_size = 2 ** i
            conv_block = TemporalConvBlock(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                dilation=dilation_size,
                padding=(kernel_size - 1) * dilation_size,
                dropout=dropout
            )
            layers.append(conv_block)
                

        self.encoder = nn.Sequential(*layers)
        self.decoder = TemporalConvNetDecoder(out_features[-1], horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        encoded = self.encoder(x)
        # Only select the final timestep feature maps
        # for forecasting
        return self.decoder(encoded[:, :, -1])

In [ ]:
batch_size = 32
in_seq_length = 35
in_features = 1

out_features = [16, 32, 64]
out_seq_length = 23


model = TemporalConvNet(
    in_features=in_features,
    out_features=out_features,
    horizon=out_seq_length,
)


in_ = torch.randn(batch_size, in_seq_length, in_features)
in_ = in_.permute(0, 2, 1)

out_ = model(in_)


In [ ]:
n_timesteps = 1000
period = 24
timesteps = np.arange(n_timesteps)
timeseries = np.sin(2 * np.pi * timesteps / period)

plt.plot(timesteps, timeseries)

In [ ]:
def prepare_single_horizon_train_dataset(
    timeseries: np.ndarray,
    in_seq_length: int,
    out_seq_length: int,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Prepares input–output pairs for single-horizon time series forecasting.

    This function constructs overlapping input sequences (features) and corresponding
    output sequences (labels) from a continuous time series. Each input window of 
    length `in_seq_length` is paired with the immediately following `out_seq_length` 
    values as the forecast target.

    :param timeseries (np.ndarray): The full time series array of shape (T,) or (T, 1), where T is the total number of timesteps.
    :param in_seq_length (int): Length of each input sequence (number of past timesteps used as context).
    :param out_seq_length (int): Number of future timesteps to predict (forecast horizon).
    """

    features, labels = [], []
    max_ts_index = len(timeseries) - in_seq_length - out_seq_length + 1
    for i in range(max_ts_index):
        feat_start, feat_end = i, i + in_seq_length
        feat_seq = timeseries[feat_start: feat_end]
        features.append(feat_seq)
        
        labels_start = i + in_seq_length
        labels_end = labels_start + out_seq_length
        labels_seq = timeseries[labels_start: labels_end]
        labels.append(labels_seq)

    features_ts = torch.tensor(np.array(features), dtype=torch.float)
    features_ts = features_ts.view(-1, in_seq_length, in_features)

    labels_ts = torch.tensor(np.array(labels), dtype=torch.float32)
    labels_ts = labels_ts.view(-1, out_seq_length, in_features)

    return features_ts, labels_ts


def prepare_multi_horizon_train_dataset(
    timeseries: np.ndarray,
    in_seq_length: int,
    out_seq_length: int,
    in_features: int = 1,
) -> tuple[np.ndarray, np.ndarray]:
    
    features, labels = [], []
    max_ts_index = len(timeseries) - in_seq_length - out_seq_length + 1
    for i in range(max_ts_index):
        feat_start, feat_end = i, i + in_seq_length
        feat_seq = timeseries[feat_start: feat_end]
        features.append(feat_seq)
        for j in range(in_seq_length):
            labels_start = i + j + 1
            labels_end = labels_start + out_seq_length
            labels_seq = timeseries[labels_start: labels_end]
            labels.append(labels_seq)
    
    features_ts = torch.tensor(np.array(features), dtype=torch.float)
    features_ts = features_ts.view(-1, in_seq_length, in_features)

    labels_ts = torch.tensor(np.array(labels), dtype=torch.float32)
    labels_ts = labels_ts.view(-1, in_seq_length, out_seq_length)

    return features_ts, labels_ts

In [ ]:
in_seq_length = 2 * period
out_seq_length = period
X, y = prepare_single_horizon_train_dataset(
    timeseries=timeseries,
    in_seq_length=in_seq_length,
    out_seq_length=out_seq_length,
)

train_ds = TensorDataset(X, y)
train_dl = DataLoader(train_ds, batch_size=32)

In [ ]:
model = TemporalConvNet(
    in_features=1,
    out_features=[16, 32, 64],
    horizon=out_seq_length,
    kernel_size=2,
    dropout=0.2
)
loss_fn = nn.MSELoss()
optimizer = AdamW(model.parameters(), lr=1e-03)

model.train()
epoch_loss, batch_loss = [], []
n_epochs = 50
pgbar = tqdm(range(n_epochs))
for epoch in pgbar:
    for batch_X, batch_y in train_dl:
        optimizer.zero_grad()
        
        # Typically sequence modelling have (batch_size, in_seq_length, num_input_fetures)
        # But CNNs work with (batch_size, num_input_features, in_seq_length) convention
        batch_X = batch_X.permute(0, 2, 1)
        y_hat = model(batch_X)

        # Output dimension is (batch_size, out_seq_length)
        y_hat = y_hat.unsqueeze(-1)
        loss = loss_fn(batch_y, y_hat)
        loss.backward()
        optimizer.step()
        
        loss_detach = float(loss.detach())
        batch_loss.append(loss_detach)
    
    epoch_loss.append(loss_detach)
    pgbar.set_description(f"Epoch [{epoch + 1} / {n_epochs}] - Loss = {loss_detach:.3f}")

In [ ]:
plt.plot(batch_loss)

In [ ]:
plt.plot(epoch_loss)

In [ ]:
# Forecast
X_test = timeseries[-in_seq_length - out_seq_length: - out_seq_length]
X_test = torch.tensor(X_test, dtype=torch.float).view(-1, in_seq_length, 1)

y_test = timeseries[-out_seq_length: ]

model.eval()
y_pred = model(X_test.permute(0, 2, 1))

plt.plot(y_pred.squeeze().detach().numpy())
plt.plot(y_test)

## UCI Dataset

In [ ]:
from datetime import datetime
import numpy as np
import polars as pl


from preprocess.explore import get_min_max_timestamps_by_client
from preprocess.constants import (
    UCI_CLIENTS_TO_DROP,
    UCI_CLIENTS_TO_FILTER,
    UCI_CLIENTS_TO_INTERPOLATE
)
from preprocess.filter import drop_client_timeseries, filter_client_timeseries
from preprocess.impute import interpolate_client_timeseries

In [ ]:
FREQUENCY_MINUTES = 15
UCI_DATA_PATH = "./data/LD2011_2014.txt"


In [ ]:
# Load data as polars dataframe
UCI_DF = pl.read_csv(
    UCI_DATA_PATH,
    has_header=True,
    separator=";",
    decimal_comma=True,
    try_parse_dates=True,
    infer_schema_length=1_000_000
)

# First column should be timestamp column
UCI_DF = UCI_DF.rename({UCI_DF.columns[0]: "timestamp"}).sort(by="timestamp")

In [ ]:
# Data processing
for client in UCI_CLIENTS_TO_DROP:
    UCI_DF = drop_client_timeseries(uci_df=UCI_DF, client_name=client)


# Filter client timeseries
for client_name, (start_ts, end_ts) in UCI_CLIENTS_TO_FILTER:
    UCI_DF = filter_client_timeseries(
        uci_df=UCI_DF,
        client_name=client_name,
        start_ts=start_ts,
        end_ts=end_ts
    )


# Interpolate client timeseries
min_max_timestamp_by_client = get_min_max_timestamps_by_client(UCI_DF)
for client_name in UCI_CLIENTS_TO_INTERPOLATE:
    # Get min / max timestamps for this client
    client_min_max_ts = min_max_timestamp_by_client.filter(pl.col("client") == client_name)
    [client_min_ts] = client_min_max_ts["min_timestamp"].to_list()
    [client_max_ts] = client_min_max_ts["max_timestamp"].to_list()

    UCI_DF = interpolate_client_timeseries(
        uci_df=UCI_DF,
        client_name=client_name,
        start_ts=client_min_ts,
        end_ts=client_max_ts,
        interval=f"{FREQUENCY_MINUTES}m"
    )

In [ ]:
# Sample some clients to forecast for.

clients_sample = (
    # Only sample from clients where we have at least 3 years of data
    min_max_timestamp_by_client
    .filter(
        pl.col("min_timestamp") <= datetime(2012, 1, 1),
        pl.col("max_timestamp") >= datetime(2015, 1, 1)
    )
    ["client"]
    .sample(n=20, seed=42)
    .to_list()
)

In [ ]:
long_sample_clients_demand_table = (
    UCI_DF
    .unpivot(
        on=[c for c in UCI_DF.columns if c != "timestamp"],
        index="timestamp",
        variable_name="client",
        value_name="demand"
    )
    .join(
        other=min_max_timestamp_by_client,
        on="client",
        how="left",
    )
    .with_columns(
        in_range=pl.col("timestamp").is_between(pl.col("min_timestamp"), pl.col("max_timestamp")),
        log1p_demand=pl.col("demand").log1p(),
    )
    .filter(
        pl.col("in_range"),
        pl.col("client").is_in(clients_sample)
    )
)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
start_ts, end_ts = datetime(2012, 10, 1), datetime(2013, 1, 1)

# --- Consumption data for each individual client ---

client = clients_sample[19]
client_df = (
    long_sample_clients_demand_table
    .filter(
        pl.col("client") == client,
        pl.col("timestamp").is_between(start_ts, end_ts)
    )
    .select(pl.col("timestamp"), pl.col("demand"), pl.col("log1p_demand"))
)
for i, col in enumerate(["demand", "log1p_demand"]):
    ax[i].plot(
        client_df["timestamp"].to_list(),
        client_df[col].to_list(),
        # alpha=0.75,
        color="tab:blue",
        lw=1.0,
    )


In [ ]:
client_df = (
    long_sample_clients_demand_table
    .filter(pl.col("client") == client)
    .select(
        pl.col("timestamp"),
        pl.col("client"),
        pl.col("demand"),
        pl.col("log1p_demand")
    )
)
bc_demand, bc_lambda = boxcox(client_df["demand"].to_numpy())

fig, ax = plt.subplots(1, 4, figsize=(20, 3.5))

ax[0].hist(client_df["demand"].to_numpy(), bins=100);
ax[1].hist(client_df["log1p_demand"].to_numpy(), bins=100);
ax[2].hist(bc_demand, bins=100);
ax[2].set_title(bc_lambda)

demand_np = client_df["demand"].to_numpy()
demand_np = (demand_np - demand_np.mean()) / demand_np.std()
ax[3].hist(demand_np, bins=100);

fig.tight_layout();

In [ ]:
# -> Use standard scaling for now.